# 1 - Datasets and differential expression

Downloads the nine GEO series of the corpus, assigns every sample to the injury or the
control group, and compares the two groups within each series (Sections 4.1-4.2).

| Output | Use |
|---|---|
| `results/DE_<series>.csv` | one table per series, read by notebook 2 |
| `results/Table1_datasets.csv` | Table 1 |
| `results/geo_identification.csv`, `results/geo_corpus_coverage.csv`, `results/geo_identified_series.csv` | Figure 1 |

Downloads are cached under `data/`, so a second run does not query GEO again.

In [1]:
# Move to the project root (the folder that contains run_all.ps1), so that relative
# paths work whether the notebook is started from notebook/ or from the root.
root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
while (!file.exists(file.path(root, "run_all.ps1")) && dirname(root) != root) {
  root <- dirname(root)
}
if (!file.exists(file.path(root, "run_all.ps1"))) {
  stop("project root not found: no run_all.ps1 above ", getwd())
}
setwd(root)
.libPaths(c(file.path(root, "env", "rlib"), .libPaths()))
Sys.setenv(RENAL_IRI_ROOT = root)
options(stringsAsFactors = FALSE, warn = 1)
cat(R.version.string, "\nroot: ", root, "\n", sep = "")

R version 4.4.3 (2025-02-28)
root: /Users/robertocasale/Desktop/lavoro_FBF/renal_shield_perfusion


## Setup

Shared settings (`R/helpers.R`), and the dataset loaders with the differential-expression
function (`R/pipeline_de.R`).

In [2]:
source("R/helpers.R")
source("R/pipeline_de.R")

[2026-10-07 16:54:20] INFO  helpers.R loaded; constants and robust_fetch() ready.



## Identification

The GEO query of Section 4.1 and the number of series it returns. The answer is cached,
because the count can change over time.

In [3]:
n_identified <- geo_identification_search()
cov <- geo_corpus_coverage()
n_q <- sum(cov$returned_by_query[cov$role == "corpus"])
cat(sprintf("%d series returned by the declared query\n", n_identified))
cat(sprintf("%d of the %d analysed series are among them; %d were identified separately\n",
            n_q, length(REGISTRY), length(REGISTRY) - n_q))
print(cov)

[2026-10-07 16:54:28] INFO  CACHE HIT   geo_identification_726da479 (queried 2026-09-11 18:41:48)



[2026-10-07 16:54:28] INFO  GEO identification: 122 series match the declared query



[2026-10-07 16:54:28] INFO  CACHE HIT   geo_serieslist_726da479_retmax500_summary (queried 2026-09-13 17:43:13)



[2026-10-07 16:54:28] INFO  WROTE       results/geo_identified_series.csv (122 rows)



[2026-10-07 16:54:28] INFO  corpus coverage: 6 of 9 analysed series returned by the declared query; 3 identified separately



122 series returned by the declared query


6 of the 9 analysed series are among them; 3 were identified separately


   accession     role returned_by_query
1   GSE30718   corpus             FALSE
2   GSE43974   corpus             FALSE
3  GSE126805   corpus             FALSE
4   GSE39548   corpus              TRUE
5   GSE98622   corpus              TRUE
6   GSE34351   corpus              TRUE
7   GSE27274   corpus              TRUE
8   GSE58438   corpus              TRUE
9  GSE148420   corpus              TRUE
10 GSE267650 held-out              TRUE


## Differential expression

One table per series, with the gene symbols of the original platform; the mapping to
human orthologs is done in notebook 2. The column `sig` (BH-adjusted p < 0.05 and
|log2 fold change| >= 0.585) is descriptive only: notebook 2 ranks all genes by p-value.

In [4]:
de <- list(); summ <- list()
for (acc in names(REGISTRY)) {
  obj <- REGISTRY[[acc]]()
  d <- run_de(obj)
  d$accession <- acc; d$organism <- obj$organism; d$modality <- obj$modality
  de[[acc]] <- d
  save_csv(d, sprintf("DE_%s.csv", acc))
  summ[[acc]] <- data.frame(
    accession = acc, organism = obj$organism, platform = obj$platform,
    n = nrow(obj$meta), n_control = sum(obj$meta$group == "control"),
    n_IRI = sum(obj$meta$group == "IRI"), paired = isTRUE(obj$paired),
    genes = nrow(d), n_sig = sum(d$sig), contrast = obj$contrast)
  log_msg(sprintf("%-10s %-18s %6d genes, %5d flagged", acc, obj$organism, nrow(d), sum(d$sig)))
}
tab1 <- do.call(rbind, summ); rownames(tab1) <- NULL
save_csv(tab1, "Table1_datasets.csv")
print(tab1[, c("accession","organism","n","n_control","n_IRI","paired","genes","n_sig")],
      row.names = FALSE)

[2026-10-07 16:54:30] INFO  CACHE HIT   geo_probe_GSE30718 (queried 2026-09-11 18:42:10)



[2026-10-07 16:54:31] INFO  GSE30718: dropped 2 repeat biopsies from 2 patients (earliest kept per GEO)



[2026-10-07 16:54:32] INFO  WROTE       results/DE_GSE30718.csv (22880 rows)



[2026-10-07 16:54:32] INFO  GSE30718   Homo sapiens        22880 genes,   374 flagged



[2026-10-07 16:54:35] INFO  CACHE HIT   geo_probe_GSE43974 (queried 2026-09-11 18:43:27)



...

[2026-10-07 17:00:55] INFO  CACHE HIT   geo_probe_GSE148420 (queried 2026-09-11 18:49:17)



[2026-10-07 17:00:56] INFO  WROTE       results/DE_GSE148420.csv (19297 rows)



[2026-10-07 17:00:56] INFO  GSE148420  Rattus norvegicus   19297 genes,  3584 flagged



[2026-10-07 17:00:56] INFO  WROTE       results/Table1_datasets.csv (9 rows)



 accession          organism   n n_control n_IRI paired genes n_sig
  GSE30718      Homo sapiens  37        11    26  FALSE 22880   374
  GSE43974      Homo sapiens 332       163   169  FALSE 34694   179
 GSE126805      Homo sapiens  82        41    41   TRUE 16862   722
  GSE39548      Mus musculus   8         4     4  FALSE 14330  3629
  GSE98622      Mus musculus  21         6    15  FALSE 29325  1028
  GSE34351      Mus musculus   6         3     3  FALSE 21495  1049
  GSE27274 Rattus norvegicus  24         6    18  FALSE 21791  1866
  GSE58438 Rattus norvegicus  19         5    14  FALSE 18784  1074
 GSE148420 Rattus norvegicus  12         4     8  FALSE 19297  3584


Next: `02_analysis.ipynb`.